# Loads and Shunts

Loads are `EnergyConsumer`s, with an optional voltage dependence (`LoadResponseCharacteristic`) and time
series (`EnergyConnectionSchedule`). Capacitor and reactor banks are `LinearShuntCompensator`s, optionally
switched by a `RegulatingControl`.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    BaseVoltageBuilder,
    EnergyConnectionScheduleBuilder,
    EnergyConsumerBuilder,
    EnergyConsumerPhaseBuilder,
    LinearShuntCompensatorBuilder,
    LinearShuntCompensatorPhaseBuilder,
    LoadResponseCharacteristicBuilder,
    RegulatingControlBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models",
    model="loads_and_shunts",
    Feeder="fdr_demo",
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_12_47kV", Qty(12.47, "kV")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

## A balanced load

`add_connectivity` gives the load its Terminal and, with `phases`, one `EnergyConsumerPhase` per phase. A
delta-connected load's phases are the phase pairs it sits between. `add_ssh` sets the load's three-phase
power.

In [2]:
load_builder = EnergyConsumerBuilder(network, container)
load = load_builder.create(mRID="ld_671", name="671")
load, terminal, load_phases = load_builder.add_connectivity(
    load, node="671", phaseConnection="D", phases="ABC", BaseVoltage="bv_12_47kV"
)
load_builder.add_ssh(load, p=Qty(1155, "kW"), q=Qty(660, "kVAr"))

print(
    load.phaseConnection.value,
    [phase.mRID for phase in load_phases],
    f"{load.p.to('kW'):g} kW",
)

D ['ld_671_AB', 'ld_671_BC', 'ld_671_AC'] 1155 kW


## An unbalanced load

For a load that differs by phase, set each `EnergyConsumerPhase` with `EnergyConsumerPhaseBuilder`. The
phases made by `add_connectivity` are named `<load>_<phase>`.

In [3]:
load = load_builder.create(mRID="ld_634", name="634")
load_builder.add_connectivity(
    load,
    node="634",
    phaseConnection="Y",
    phases="ABC",
    grounded=True,
    BaseVoltage="bv_12_47kV",
)
phase_builder = EnergyConsumerPhaseBuilder(network, container)
for phase, p, q in [("A", 160, 110), ("B", 120, 90), ("C", 120, 90)]:
    phase_builder.add_ssh(
        network.find_by_mRID(f"ld_634_{phase}"), p=Qty(p, "kW"), q=Qty(q, "kVAr")
    )
load_builder.add_ssh(load, p=Qty(400, "kW"), q=Qty(290, "kVAr"))

for phase in sorted(load.EnergyConsumerPhase, key=lambda phase: phase.mRID):
    print(f"{phase.mRID}: {phase.p.to('kW'):g} kW, {phase.q.to('kVAr'):g} kVAr")

ld_634_A: 160 kW, 110 kVAr
ld_634_B: 120 kW, 90 kVAr
ld_634_C: 120 kW, 90 kVAr


## Voltage dependence

A `LoadResponseCharacteristic` says how a load's power varies with voltage, either as ZIP fractions of
constant impedance, current and power, or as voltage exponents. Any number of loads can share one. This
one is constant impedance:

In [4]:
response_builder = LoadResponseCharacteristicBuilder(network, container)
response = response_builder.create(mRID="lrc_constant_z", name="constant Z")
response_builder.add_electrical(
    response,
    exponentModel=False,
    pConstantImpedance=1.0,
    pConstantCurrent=0.0,
    pConstantPower=0.0,
    qConstantImpedance=1.0,
    qConstantCurrent=0.0,
    qConstantPower=0.0,
)
load_builder.add_electrical(
    network.find_by_mRID("ld_671"), LoadResponse="lrc_constant_z"
)

print([load.mRID for load in response.EnergyConsumer])

['ld_671']


## Load profiles

An `EnergyConnectionSchedule` attaches a time series to a load (or any other `EnergyConnection`). The
schedule itself is a graph object. Its time points are not: `time_points` returns them as a PyArrow table,
in SI units, in UTC and sorted by time, ready to write to Parquet next to the model.

In [5]:
from datetime import datetime, timedelta

schedule_builder = EnergyConnectionScheduleBuilder(network, container)
schedule = schedule_builder.create(mRID="ld_671_profile", name="671 daily")
schedule_builder.add_schedule(
    schedule,
    EnergyConnection="ld_671",
    timeSeriesKind="schedule",
    interpolationKind="previous",
)

hours = [datetime(2026, 7, 1) + timedelta(hours=hour) for hour in range(0, 24, 6)]
table = schedule_builder.time_points(
    schedule,
    times=hours,
    p=[700, 900, 1155, 1000],
    q=[400, 520, 660, 570],
    p_unit="kW",
    q_unit="kVAr",
)
for row in table.to_pylist():
    print(
        f"{row['atTime']:%Y-%m-%d %H:%M}",
        f"p = {row['p']:9.0f} W",
        f"q = {row['q']:8.0f} VAr",
    )

2026-07-01 00:00 p =    700000 W q =   400000 VAr
2026-07-01 06:00 p =    900000 W q =   520000 VAr
2026-07-01 12:00 p =   1155000 W q =   660000 VAr
2026-07-01 18:00 p =   1000000 W q =   570000 VAr


## A capacitor bank

`add_connectivity` takes the `compensatorType`, `capacitor` or `reactor`. `add_electrical` takes the
rated voltage and either the reactive power of the whole bank, `nomQ`, from which it works out the
susceptance of each of its `maximumSections` sections, or `bPerSection` itself. `add_ssh` sets how many
sections are in service. This is a 600 kVAr bank switched in three 200 kVAr steps.

In [6]:
cap_builder = LinearShuntCompensatorBuilder(network, container)
cap = cap_builder.create(mRID="cap_675", name="675")
cap, cap_terminal, cap_phases = cap_builder.add_connectivity(
    cap,
    node="675",
    compensatorType="capacitor",
    phaseConnection="Y",
    phases="ABC",
    grounded=True,
    BaseVoltage="bv_12_47kV",
)
cap_builder.add_electrical(
    cap,
    nomU=Qty(12.47, "kV"),
    nomQ=Qty(600, "kVAr"),
    normalSections=3,
    maximumSections=3,
)
cap_builder.add_ssh(cap, sections=3)

print(f"bPerSection = {cap.bPerSection.to('uS'):.1f} µS")
print(f"{cap.sections:g} of {cap.maximumSections} sections in service")

bPerSection = 1286.2 µS
3 of 3 sections in service


Each phase can be switched separately too: `LinearShuntCompensatorPhaseBuilder.add_ssh` sets the sections
in service on each `<capacitor>_<phase>` object. Here phase C is switched out:

In [7]:
cap_phase_builder = LinearShuntCompensatorPhaseBuilder(network, container)
for phase in "ABC":
    cap_phase_builder.add_ssh(
        network.find_by_mRID(f"cap_675_{phase}"), sections=0 if phase == "C" else 3
    )

print(
    [
        (phase.mRID, f"{phase.sections:g}")
        for phase in sorted(cap.ShuntCompensatorPhase, key=lambda p: p.mRID)
    ]
)

[('cap_675_A', '3'), ('cap_675_B', '3'), ('cap_675_C', '0')]


## Voltage control

A switched capacitor needs a `RegulatingControl`: what it watches, here the voltage at its own Terminal,
and the band it holds that to. Like a tap changer control, the target and deadband are in primary volts,
here 120.5 V ± 1.5 V on a 60:1 potential transformer. `add_regulating_control` on the capacitor then links
the two and sets the switching delay.

In [8]:
control_builder = RegulatingControlBuilder(network, container)
control = control_builder.create(mRID="cap_675_ctrl")
control_builder.add_regulating_control(
    control,
    Terminal=cap_terminal.mRID,
    mode="voltage",
    targetValue=120.5 * 60,
    targetDeadband=3 * 60,
    ptRatio=60,
    discrete=True,
    enabled=True,
)
cap_builder.add_regulating_control(
    cap, RegulatingControl=control.mRID, controlEnabled=True, aVRDelay=Qty(30, "s")
)

print(
    f"{cap.RegulatingControl.mode.value} control, target {control.targetValue:g} V, "
    f"deadband {control.targetDeadband:g} V, delay {cap.aVRDelay.to('s'):g} s"
)

voltage control, target 7230 V, deadband 180 V, delay 30 s
